In [2]:
!pip install pandas numpy scikit-learn openpyxl xlrd joblib

In [3]:
# ==========================================
# CELL 1: IMPORTS & ENVIRONMENT SETUP
# ==========================================
import os
import pandas as pd
import numpy as np
import warnings
import joblib

# Scikit-Learn Machine Learning Tools
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Suppress messy terminal warnings
warnings.filterwarnings('ignore') 

print("Cell 1 Complete: Environment Ready! Libraries successfully loaded.")

Cell 1 Complete: Environment Ready! Libraries successfully loaded.


In [4]:
# ==========================================
# CELL 2: DATA EXTRACTION & VECTORIZED MATH
# ==========================================

def process_arbin_standard(file_path, sheets, temp):
    """Processes standard Incremental OCV tests using Option B scaling."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")
        
    print(f"Processing standard data at {temp}°C: {file_path}")
    dfs = [pd.read_excel(file_path, sheet_name=sheet) for sheet in sheets]
    df = pd.concat(dfs, ignore_index=True).sort_values('Test_Time(s)').reset_index(drop=True)
    
    # 1. Coulomb Counting: Net Capacity = Charge - Discharge
    df['Net_Capacity(Ah)'] = df['Charge_Capacity(Ah)'] - df['Discharge_Capacity(Ah)']
    
    # 2. Option B: File-Specific MIN and MAX
    min_cap = df['Net_Capacity(Ah)'].min()
    max_cap = df['Net_Capacity(Ah)'].max()
    
    # 3. Vectorized SOC Calculation
    df['SOC'] = (df['Net_Capacity(Ah)'] - min_cap) / (max_cap - min_cap)
    
    # 4. The Firewall: Return ONLY what the real car knows + the Answer Key
    return pd.DataFrame({
        'Time(s)': df['Test_Time(s)'],
        'Voltage(V)': df['Voltage(V)'],
        'Current(A)': df['Current(A)'],
        'Temperature(C)': temp,
        'SOC': df['SOC']
    })

def process_low_current(file_path, sheet, temp, has_temp_col=False):
    """Processes low-current tests using Calculus Integration & Option B scaling."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")
        
    print(f"Processing low-current data at {temp}°C: {file_path}")
    df = pd.read_excel(file_path, sheet_name=sheet).rename(columns={'Duration (sec)': 'Time(s)'})
    
    for col in ['Time(s)', 'mV', 'mA']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['Time(s)', 'mV', 'mA'])
    
    df['Temperature(C)'] = pd.to_numeric(df['Temperature'], errors='coerce').ffill().fillna(temp) if has_temp_col and 'Temperature' in df.columns else temp
        
    df['Voltage(V)'] = df['mV'] / 1000.0
    df['Current(A)'] = df['mA'] / 1000.0
    
    # 1. Coulomb Counting: Integral of (Current * dt) / 3600
    dt = df['Time(s)'].diff().fillna(df['Time(s)'].iloc[0])
    df['Net_Capacity(Ah)'] = (df['Current(A)'] * dt / 3600.0).cumsum()
    
    # 2. Option B: File-Specific MIN and MAX
    min_cap = df['Net_Capacity(Ah)'].min()
    max_cap = df['Net_Capacity(Ah)'].max()
    
    # 3. Vectorized SOC Calculation
    df['SOC'] = (df['Net_Capacity(Ah)'] - min_cap) / (max_cap - min_cap)
    
    # 4. The Firewall
    return pd.DataFrame({
        'Time(s)': df['Time(s)'],
        'Voltage(V)': df['Voltage(V)'],
        'Current(A)': df['Current(A)'],
        'Temperature(C)': df['Temperature(C)'],
        'SOC': df['SOC']
    })

print("Cell 2 Complete: Physics engines loaded and firewalls established.")

Cell 2 Complete: Physics engines loaded and firewalls established.


In [5]:
# ==========================================
# CELL 3: BUILDING THE MASTER DATASET
# ==========================================

dataset_path = 'Master_Battery_Data_Clean.csv'

if os.path.exists(dataset_path):
    print(f"Found existing '{dataset_path}'. Loading directly into memory...")
    master_data = pd.read_csv(dataset_path)
    print(f"Loaded {master_data.shape[0]} records successfully!")
else:
    print("Building Master Dataset from raw Excel files (This may take ~30-60 seconds)...")
    datasets = []
    
    # 0°C Data
    datasets.append(process_arbin_standard('data/02_26_2016_SP20-1_0C_incrementalOCV.xls', ['Channel_1-005_1', 'Channel_1-005_2', 'Channel_1-005_3'], 0))
    datasets.append(process_arbin_standard('data/02_24_2016_SP20-1_0C_lowcurrentOCV.xls', ['Channel_1-005_1', 'Channel_1-005_2'], 0))
    
    # 25°C Data
    datasets.append(process_arbin_standard('data/10_16_2015_Initial capacity_SP20-1.xlsx', ['Sheet1'], 25))
    datasets.append(process_low_current('data/11_5_2015_low current OCV test_SP20-1.xlsx', 'SP20-OCVSOC-0.05C', 25, True))
    datasets.append(process_arbin_standard('data/12_2_2015_Incremental OCV test_SP20-1.xlsx', ['Channel_1-005_1', 'Channel_1-005_2', 'Channel_1-005_3'], 25))

    # 45°C Data
    datasets.append(process_low_current('data/11_21_2015_low current OCV test_SP20-1.xlsx', '11_21_2015_45C_SP20-1_samallcur', 45, False))
    datasets.append(process_arbin_standard('data/12_09_2015_Incremental OCV test_SP20-1.xlsx', ['Channel_1-005_1', 'Channel_1-005_2', 'Channel_1-005_3'], 45))

    # Concatenate all datasets and remove broken rows
    master_data = pd.concat(datasets, ignore_index=True).dropna()
    
    # Save a clean backup to your hard drive
    master_data.to_csv(dataset_path, index=False)
    print(f"\nCell 3 Complete: Master Dataset built with {master_data.shape[0]} records and saved to disk.")

# Display a quick preview to prove Capacity(Ah) is GONE!
master_data.head()

Building Master Dataset from raw Excel files (This may take ~30-60 seconds)...
Processing standard data at 0°C: data/02_26_2016_SP20-1_0C_incrementalOCV.xls
Processing standard data at 0°C: data/02_24_2016_SP20-1_0C_lowcurrentOCV.xls
Processing standard data at 25°C: data/10_16_2015_Initial capacity_SP20-1.xlsx
Processing low-current data at 25°C: data/11_5_2015_low current OCV test_SP20-1.xlsx
Processing standard data at 25°C: data/12_2_2015_Incremental OCV test_SP20-1.xlsx
Processing low-current data at 45°C: data/11_21_2015_low current OCV test_SP20-1.xlsx
Processing standard data at 45°C: data/12_09_2015_Incremental OCV test_SP20-1.xlsx

Cell 3 Complete: Master Dataset built with 904763 records and saved to disk.


,Time(s),Voltage(V),Current(A),Temperature(C),SOC
0,10.000019,4.163778,0.0,0.0,0.988246
1,20.015468,4.163778,0.0,0.0,0.988246
2,30.030954,4.163778,0.0,0.0,0.988246
3,40.046428,4.163778,0.0,0.0,0.988246
4,50.061910,4.163940,0.0,0.0,0.988246


In [6]:
# ==========================================
# CELL 4: DATA SPLITTING & RANDOM FOREST
# ==========================================

print("--- Preparing Data (Fixing Data Leakage) ---")
# Sample 100,000 random points representing all temperatures
df_sample = master_data.sample(n=100000, random_state=42)

# THE REAL WORLD INPUTS: Only what a car sensor can see. NO Capacity!
X = df_sample[['Voltage(V)', 'Current(A)', 'Temperature(C)', 'Time(s)']]

# THE TARGET: What the AI needs to guess
y = df_sample['SOC']

# Hide 20% of the data to test the AI later
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data Split Complete: 80,000 Training rows, 20,000 Testing rows.")

print("\n--- Training Random Forest (Learning Non-Linear Physics) ---")
# Build 50 decision trees, limit depth to 15 to prevent overfitting
rf_model = RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Make predictions on the hidden 20%
preds_rf = rf_model.predict(X_test)

print("\n🏆 RANDOM FOREST EVALUATION SCORES")
print(f"MAE:  {mean_absolute_error(y_test, preds_rf):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds_rf)):.4f}")
print(f"R²:   {r2_score(y_test, preds_rf):.4f}")

# Save the model for future deployment
joblib.dump(rf_model, 'battery_soc_rf_clean.pkl')
print("\n✅ Random Forest Model saved successfully!")

--- Preparing Data (Fixing Data Leakage) ---
Data Split Complete: 80,000 Training rows, 20,000 Testing rows.

--- Training Random Forest (Learning Non-Linear Physics) ---

🏆 RANDOM FOREST EVALUATION SCORES
MAE:  0.0003
RMSE: 0.0035
R²:   0.9999

✅ Random Forest Model saved successfully!


In [7]:
# ==========================================
# CELL 5: LINEAR REGRESSION (The Baseline)
# ==========================================

print("--- Training Linear Regression (Testing for Non-Linearity) ---")
# Build a basic linear model (y = mx + b)
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)

# Make predictions on the hidden 20%
preds_lr = lin_model.predict(X_test)

print("\n📉 LINEAR REGRESSION EVALUATION SCORES")
print(f"MAE:  {mean_absolute_error(y_test, preds_lr):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds_lr)):.4f}")
print(f"R²:   {r2_score(y_test, preds_lr):.4f}")
print("\nNotice how this score is significantly lower than the Random Forest!")
print("This proves the battery physics are non-linear, and we NEED complex AI.")

--- Training Linear Regression (Testing for Non-Linearity) ---

📉 LINEAR REGRESSION EVALUATION SCORES
MAE:  0.0712
RMSE: 0.0981
R²:   0.8935

Notice how this score is significantly lower than the Random Forest!
This proves the battery physics are non-linear, and we NEED complex AI.


In [8]:
# ==========================================
# CELL 6: XGBOOST (Gradient Boosting)
# ==========================================
print("--- Training XGBoost (High-Speed Production Model) ---")

# Build trees sequentially to correct previous errors
xgb_model = HistGradientBoostingRegressor(max_iter=200, random_state=42)
xgb_model.fit(X_train, y_train)

# Make predictions on the hidden 20%
preds_xgb = xgb_model.predict(X_test)

print("\n🚀 XGBOOST EVALUATION SCORES")
print(f"MAE:  {mean_absolute_error(y_test, preds_xgb):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds_xgb)):.4f}")
print(f"R²:   {r2_score(y_test, preds_xgb):.4f}")

# Save the model
joblib.dump(xgb_model, 'battery_soc_xgb_clean.pkl')
print("\n✅ XGBoost Model saved successfully!")

--- Training XGBoost (High-Speed Production Model) ---

🚀 XGBOOST EVALUATION SCORES
MAE:  0.0026
RMSE: 0.0053
R²:   0.9997

✅ XGBoost Model saved successfully!


In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler

In [12]:
# ==========================================
# CELL 7: ARTIFICIAL NEURAL NETWORK (EPOCHS)
# ==========================================
# Note: If tensorflow is not installed, run !pip install tensorflow in an empty cell first
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler

print("--- Preparing Data for Deep Learning ---")
# Neural Networks require scaled data to converge properly
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from tensorflow.keras.callbacks import EarlyStopping

print("\n--- Training the ANN (With Early Stopping) ---")

# 1. Create the Safety Net
early_stop = EarlyStopping(
    monitor='val_loss',       # Watch the hidden 20% test data
    patience=10,              # If it doesn't improve for 10 epochs, stop!
    restore_best_weights=True # Automatically rewind to the best version of the brain
)

# 2. Run your massive epochs safely!
history = ann_model.fit(
    X_train_scaled, y_train,
    epochs=300000,            # You can keep your massive number!
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],   # <--- Attach the safety net here
    verbose=1
)

print("\n🧠 ANN EVALUATION SCORES")
preds_ann = ann_model.predict(X_test_scaled).flatten()
print(f"MAE:  {mean_absolute_error(y_test, preds_ann):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds_ann)):.4f}")
print(f"R²:   {r2_score(y_test, preds_ann):.4f}")

# Save the Deep Learning model and its specific scaler
ann_model.save('battery_soc_ann.keras')
joblib.dump(scaler, 'ann_scaler.pkl')
print("\n✅ ANN Model and Scaler saved successfully!")

--- Preparing Data for Deep Learning ---

--- Training the ANN (With Early Stopping) ---
Epoch 1/300000
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 9.7573e-05 - mae: 0.0064 - val_loss: 5.3262e-04 - val_mae: 0.0183
Epoch 2/300000
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 9.3440e-05 - mae: 0.0062 - val_loss: 5.5566e-04 - val_mae: 0.0183
Epoch 3/300000
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 9.0823e-05 - mae: 0.0062 - val_loss: 8.3324e-04 - val_mae: 0.0223
Epoch 4/300000
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 8.9597e-05 - mae: 0.0061 - val_loss: 5.8847e-04 - val_mae: 0.0188
Epoch 5/300000
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 8.8779e-05 - mae: 0.0061 - val_loss: 5.0095e-04 - val_mae: 0.0180
Epoch 6/300000
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 8.7667e-05 - mae: 0.0060 - val_loss: 4.9799e-04 - val_mae: 0.0179
Epoch 7/300000
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 8.3657e-05 - mae: 0.0059 - val_loss: 5.5877e-04 - val

In [14]:
# ==========================================
# CELL 8: FLAWLESS TIME-SERIES LSTM MODEL
# ==========================================
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

print("--- 1. Preparing Pure Sequential Sequences ---")
# Safe downsampling to preserve RAM while maintaining physics curves
df_lstm = master_data.iloc[::5, :].reset_index(drop=True)

# Create a unique ID tracker based on where Temperature changes or Time resets
# This isolates our 7 original experiments from each other!
df_lstm['Experiment_ID'] = (df_lstm['Time(s)'].diff() < 0).cumsum()

# Scale the sensors globally across the dataset
scaler_lstm = StandardScaler()
features = ['Voltage(V)', 'Current(A)', 'Temperature(C)', 'Time(s)']
df_lstm[features] = scaler_lstm.fit_transform(df_lstm[features])

# The Corrected Windowing Function: Stops windows from bleeding across files
def create_isolated_sequences(df, features, target_col, time_steps=10):
    Xs, ys = [], []
    # Process each experiment completely independently!
    for exp_id, group in df.groupby('Experiment_ID'):
        X_val = group[features].values
        y_val = group[target_col].values
        
        if len(X_val) <= time_steps:
            continue
            
        for i in range(len(X_val) - time_steps):
            Xs.append(X_val[i:(i + time_steps)])
            ys.append(y_val[i + time_steps])
            
    return np.array(Xs), np.array(ys)

print("Building 3D Time-Windows safely within experimental boundaries...")
X_3D, y_1D = create_isolated_sequences(df_lstm, features, 'SOC', time_steps=10)

# Sequential Split: Train on past 80%, Test on future 20%
split_idx = int(len(X_3D) * 0.8)
X_train_lstm, X_test_lstm = X_3D[:split_idx], X_3D[split_idx:]
y_train_lstm, y_test_lstm = y_1D[:split_idx], y_1D[split_idx:]

print(f"Data Pipeline Secured! Shape: {X_train_lstm.shape}")

print("\n--- 2. Building the LSTM Network ---")
lstm_model = Sequential([
    LSTM(64, return_sequences=False, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    Dropout(0.1),
    Dense(32, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("\n--- 3. Training the LSTM ---")
history = lstm_model.fit(
    X_train_lstm, y_train_lstm,
    epochs=30,
    batch_size=256,
    validation_split=0.2,
    shuffle=True, # Splitting was done safely, we can shuffle batches now!
    callbacks=[early_stop],
    verbose=1
)

print("\n--- 4. Final LSTM Evaluation ---")
preds_lstm = lstm_model.predict(X_test_lstm).flatten()
print(f"MAE:  {mean_absolute_error(y_test_lstm, preds_lstm):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_lstm, preds_lstm)):.4f}")
print(f"R²:   {r2_score(y_test_lstm, preds_lstm):.4f}")

# Save assets
lstm_model.save('battery_soc_lstm_clean.keras')
joblib.dump(scaler_lstm, 'lstm_scaler_clean.pkl')
print("\n✅ Cleaned LSTM Pipeline compiled successfully!")

--- 1. Preparing Pure Sequential Sequences ---
Building 3D Time-Windows safely within experimental boundaries...
Data Pipeline Secured! Shape: (144706, 10, 4)

--- 2. Building the LSTM Network ---

--- 3. Training the LSTM ---
Epoch 1/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - loss: 0.0053 - mae: 0.0442 - val_loss: 0.0075 - val_mae: 0.0636
Epoch 2/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 9.2849e-04 - mae: 0.0227 - val_loss: 0.0040 - val_mae: 0.0493
Epoch 3/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - loss: 6.2316e-04 - mae: 0.0186 - val_loss: 0.0037 - val_mae: 0.0497
Epoch 4/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 4.8505e-04 - mae: 0.0163 - val_loss: 0.0026 - val_mae: 0.0405
Epoch 5/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 3.9619e-04 - mae: 0.0146 - val_loss: 0.0021 - val_mae: 0.0353
Epoch 6/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - loss: 3.4123e-04 - mae: 0.0135 - val_loss: 0.0020 - val_mae: 0.0356
Epoch 7/30
453/453 ━━━━━━━━━━━━━━━━━━━━